# **心臟疾病辨識單元(一)：自動駕駛車怎麼辨認路上的狗狗和狗扁扁？**

# 0.先收錄一些會用到的公式和函式吧！

In [1]:
#@title
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import itertools

from keras.models import Model, Sequential
from keras.layers import Conv1D, GlobalAveragePooling1D, MaxPooling1D
from keras.layers import Activation, Dense, Dropout, Input, Embedding, Flatten
from keras.callbacks import CSVLogger
from tensorflow.keras.optimizers import RMSprop,Adam

from ast import literal_eval
from keras.utils.np_utils import to_categorical
import matplotlib.pyplot as plt
from keras.layers import LeakyReLU

# 1.先將7個波峰為一組的心電訊號丟進程式庫吧！

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  
df = pd.read_csv('CNN_心電訊號.csv',delimiter=',',converters=dict(v2 = literal_eval ))
df.info()

# 2.來看看加進來心電訊號吧！

In [ ]:
#讀取資料
data = np.array(df)
data2 = data[0 : , 1]

#稍微設定一下畫布資訊！
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2[0], #取用其中一筆正常時段的心電訊號
    mode='lines',
    name='Original Plot',
    
)
)
fig.update_layout(
    title="其中一筆正常時段的心電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    y=data2[35], #取用其中一筆心律不整時段的心電訊號
    mode='lines',
    name='Original Plot'
))
fig2.update_layout(
    title="其中一筆心律不整時段的心電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#秀出你的成果吧！
fig2.show()

# 3.將這些資料「補零」來固定資料長度吧！


*   觀察上面的資料長度，修改lengthOfData
*   微調直到足夠大的lengthOfData能讓程式結果 error 數為零即可
*   太大的lengthOfData對電腦來說會是很大的負擔，盡可能找到最小lengthOfData吧！

In [ ]:
X = df.iloc[:,1:].values
lengthOfData = 3000 #猜測資料中最大長度可能是幾筆                              #迷之音：還是要試試看3000呢^^
numberOfData = 60   

tempZ = np.zeros(shape=(numberOfData,lengthOfData))
tempX = []
error = 0
count=0

for a in X:
    amountOfPadding = lengthOfData - len(a[0])

    if len(a[0]) < lengthOfData :
        #該筆資料長度比設定的lengthOfData小就會進行補零
        tempX = np.pad(a[0], (0,amountOfPadding), 'constant', constant_values=(0,0))
        tempZ[count] = tempX
        count = count + 1         
    else: 
        error = error + 1 #該筆資料長度比設定的lengthOfData大就會加一個error數
        tempX = np.pad(a[0], (0,0), 'constant', constant_values=(0,0))
        count = count + 1


Y = df.iloc[:,:1].values
Y = to_categorical(Y,num_classes=2)

if error != 0 :
  print('error=',error, "試試看將lengthOfData再往上調高一些吧！")
else:
  print('error=',error, "順利完成補零(Padding)，繼續往下一步吧！")

# 4.來觀察這些「補零」後的資料長什麼樣子吧！



In [ ]:
#稍微設定一下畫布資訊！
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    y=tempZ[0], #取用其中一筆正常時段的心電訊號
    mode='lines',
    name='Original Plot',
)
)
fig3.update_layout(
    title="其中一筆「補零後」正常時段的心電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig3.show()

In [ ]:
#稍微設定一下畫布資訊！
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    y=tempZ[35], #取用其中一筆心律不整時段的心電訊號
    mode='lines',
    name='Original Plot',
)
)
fig4.update_layout(
    title="其中一筆「補零後」心律不整時段的心電訊號",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig4.show()


#5.單元小結論：
---
#1. 心律不整的訊號比起正常訊號來得不規律許多

#2. 心電訊號的資料並不像照片有固定畫素和格式
#3. 透過補零(Padding)的技巧能固定心電訊號的格式